In [1]:
import pandas as pd
import numpy as np
import json
import os
from jiwer import wer, cer
from datasets import load_dataset
import re
from num2words import num2words
from scipy.stats import iqr, ttest_rel
import matplotlib.pyplot as plt
import seaborn as sns



c:\Users\ATHO0358\Documents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Analysis of the Wrist Angel Dataset being run through the ASR module only.

In [ ]:
batches_per_epoch = 1689
batch_stats = []
with open("../results/wav2vec2_baseline.jsonl") as f:
    for batch_idx, line in enumerate(f):
        samples = json.loads(line)

        avg_duration = sum(s["seg_end"] for s in samples) / len(samples)

        batch_stats.append({
            "epoch": batch_idx // batches_per_epoch,
            "batch": batch_idx % batches_per_epoch,
            "avg_audio_duration": avg_duration,
            "n_samples": len(samples),
        })

df = pd.DataFrame(batch_stats)
df.head(20)

In [2]:
import jiwer

def evaluate_wer_components(references: list[str], hypotheses: list[str]):
    """
    Computes global WER alongside its individual error constituents:
    Substitutions (S), Insertions (I), Deletions (D), and Reference Word Count (N).
    """
    # Process transcriptions across the dataset
    word_output = jiwer.process_words(references, hypotheses)
    
    # Extract raw counts across all audio files
    w_sub = word_output.substitutions
    w_ins = word_output.insertions
    w_del = word_output.deletions
    w_total = word_output.hits + word_output.substitutions + word_output.deletions # N
    wer = word_output.wer

    char_output = jiwer.process_characters(references, hypotheses)
    
    c_sub = char_output.substitutions
    c_ins = char_output.insertions
    c_del = char_output.deletions
    c_total = char_output.hits + c_sub + c_del  # Total reference characters (N_chars)
    cer = char_output.cer
    
    print("=" * 45)
    print("           ASR ERROR EVALUATION           ")
    print("=" * 45)
    print(f"WORD LEVEL (WER: {wer:.2%})")
    print(f"  • Reference Words (N) : {wer}")
    print(f"  • Substitutions (S)   : {w_sub} ({w_sub / w_total:.2%})")
    print(f"  • Insertions (I)      : {w_ins} ({w_ins / w_total:.2%})")
    print(f"  • Deletions (D)       : {w_del} ({w_del / w_total:.2%})")
    print("-" * 45)
    print(f"CHARACTER LEVEL (CER: {cer:.2%})")
    print(f"  • Reference Chars (N) : {c_total}")
    print(f"  • Substitutions (S)   : {c_sub} ({c_sub / c_total:.2%})")
    print(f"  • Insertions (I)      : {c_ins} ({c_ins / c_total:.2%})")
    print(f"  • Deletions (D)       : {c_del} ({c_del / c_total:.2%})")
    print("=" * 45)

    return {
        "wer": wer,
        "cer": cer,
        "word_metrics": {
            "substitutions": w_sub,
            "insertions": w_ins,
            "deletions": w_del,
            "total_words": w_total,
        },
        "char_metrics": {
            "substitutions": c_sub,
            "insertions": c_ins,
            "deletions": c_del,
            "total_chars": c_total,
        }
    }

### Standard WER and CER of the CoRal Dataset using the Roest-v3-Wav2vec2-315m model 

In [22]:
references = []
hypotheses = []

with open("../results/wav2vec2_baseline.jsonl") as f:
    for line in f:
        batch = json.loads(line)
        for item in batch:
            references.append(item['ref'])
            hypotheses.append(item['hyp'])
print(f'Ref shape: {len(references)}')
print(f'Hyp length: {len(hypotheses)}')

evaluate_wer_components(references=references, hypotheses=hypotheses)

Ref shape: 25314
Hyp length: 25314
           ASR ERROR EVALUATION           
WORD LEVEL (WER: 34.30%)
  • Reference Words (N) : 0.34299239856938785
  • Substitutions (S)   : 68598 (23.84%)
  • Insertions (I)      : 11061 (3.84%)
  • Deletions (D)       : 19023 (6.61%)
---------------------------------------------
CHARACTER LEVEL (CER: 16.43%)
  • Reference Chars (N) : 1341279
  • Substitutions (S)   : 83613 (6.23%)
  • Insertions (I)      : 42108 (3.14%)
  • Deletions (D)       : 94656 (7.06%)


{'wer': 0.34299239856938785,
 'cer': 0.1643036236308777,
 'word_metrics': {'substitutions': 68598,
  'insertions': 11061,
  'deletions': 19023,
  'total_words': 287709},
 'char_metrics': {'substitutions': 83613,
  'insertions': 42108,
  'deletions': 94656,
  'total_chars': 1341279}}

### Standard WER and CER of the Wrist Angel Dataset using the Roest-v3-Wav2vec2-315m Model

In [3]:
references = []
hypotheses = []

with open("../results/wa_wav2vec2_baseline.jsonl") as f:
    for line in f:
        batch = json.loads(line)
        for item in batch:
            references.append(item['ref'])
            hypotheses.append(item['hyp'])
print(f'Ref shape: {len(references)}')
print(f'Hyp length: {len(hypotheses)}')

evaluate_wer_components(references=references, hypotheses=hypotheses)

Ref shape: 3138
Hyp length: 3138
           ASR ERROR EVALUATION           
WORD LEVEL (WER: 72.91%)
  • Reference Words (N) : 0.7290896353092282
  • Substitutions (S)   : 22208 (43.22%)
  • Insertions (I)      : 6075 (11.82%)
  • Deletions (D)       : 9182 (17.87%)
---------------------------------------------
CHARACTER LEVEL (CER: 44.69%)
  • Reference Chars (N) : 251201
  • Substitutions (S)   : 30024 (11.95%)
  • Insertions (I)      : 27683 (11.02%)
  • Deletions (D)       : 54543 (21.71%)


{'wer': 0.7290896353092282,
 'cer': 0.446853316666733,
 'word_metrics': {'substitutions': 22208,
  'insertions': 6075,
  'deletions': 9182,
  'total_words': 51386},
 'char_metrics': {'substitutions': 30024,
  'insertions': 27683,
  'deletions': 54543,
  'total_chars': 251201}}

### Standard WER and CER of the Wrist Angel Dataset using the 8-bit Quantized Roest-v3-Whisper-1.5b Model

In [10]:
references = []
hypotheses = []

with open("../results/wa_ct2_int8.jsonl") as f:
    for line in f:
        batch = json.loads(line)
        for item in batch:
            references.append(item['ref'])
            hypotheses.append(item['hyp'])
print(f'Ref shape: {len(references)}')
print(f'Hyp length: {len(hypotheses)}')

evaluate_wer_components(references=references, hypotheses=hypotheses)

Ref shape: 3513
Hyp length: 3513
           ASR ERROR EVALUATION           
WORD LEVEL (WER: 67.71%)
  • Reference Words (N) : 0.6771426926840499
  • Substitutions (S)   : 18366 (35.24%)
  • Insertions (I)      : 8652 (16.60%)
  • Deletions (D)       : 8274 (15.88%)
---------------------------------------------
CHARACTER LEVEL (CER: 45.22%)
  • Reference Chars (N) : 254385
  • Substitutions (S)   : 28140 (11.06%)
  • Insertions (I)      : 38385 (15.09%)
  • Deletions (D)       : 48510 (19.07%)


{'wer': 0.6771426926840499,
 'cer': 0.45220826699687483,
 'word_metrics': {'substitutions': 18366,
  'insertions': 8652,
  'deletions': 8274,
  'total_words': 52119},
 'char_metrics': {'substitutions': 28140,
  'insertions': 38385,
  'deletions': 48510,
  'total_chars': 254385}}

### Standard WER and CER of the Wrist Angel Dataset using the 8-bit Quantized Roest-v3-Whisper-1.5b Model of the Long Audio Recordings

In [8]:
references = {}
hypotheses = {}


with open("../data/wa_transcripts/temp_ct2_int8_results.jsonl", 'r') as read_hyp:
    for line in read_hyp:
        batch = json.loads(line)
        hyp = batch[0]['hyp']
        full_text = " ".join([item['text'] for item in hyp])
        hypotheses[batch[0]['audio_id']] = full_text


with open('../data/audio/segments/transcripts.jsonl', 'r') as read_ref:
    for line in read_ref:
        item = json.loads(line)
        id = item['audio_id']
        full_text = " ".join([seg['text'] for seg in item['segments']])
        references[id] = full_text


refs = {k: v for k, v in sorted(references.items(), key=lambda item: item[0])}
hyps = {k: v for k, v in sorted(hypotheses.items(), key=lambda item: item[0])}
print(refs.keys())
print(list(refs.values()))
print(hyps.keys())
print(list(hyps.values()))

evaluate_wer_components(references=list(refs.values()), hypotheses=list(hyps.values()))

dict_keys(['id18_baseline_suds', 'id38_baseline_exposure', 'id38_week8_exposure', 'id47_baseline_exposure2', 'id47_baseline_exposure3', 'id47_week8_exposure2', 'id47_week8_exposure3', 'id4_baseline_exposure1', 'id4_baseline_exposure2', 'id4_week8_exposure1', 'id4_week8_exposure2', 'id4_week8_exposure3', 'id50_baseline_exposure1', 'id50_baseline_exposure2', 'id50_week8_exposure1', 'id55_baseline_exposure1', 'id55_baseline_exposure2', 'id55_week8_exposure', 'id56_baseline_exposure1', 'id56_baseline_exposure2', 'id6_baseline_exposure2', 'id6_baseline_exposure3', 'id6_week8_exposure1', 'id6_week8_exposure2'])
["Og når man prøver at arbejde mod OCD'en så i virkeligheden, så skal man prøve at vælge et eller andet, man kan prøve at gøre og mærke noget ubehag. Det kan godt være lidt svært at sådan... Men når man får de her tanker, som kan være ubehagelige eller som man ikke rigtig kan slippe af med, kan du så mærke ubehaget i kroppen? Der er nogle, der kan mærke det ved, at deres puls stiger e

{'wer': 0.7010911495646424,
 'cer': 0.5386595927284076,
 'word_metrics': {'substitutions': 4477,
  'insertions': 1923,
  'deletions': 6322,
  'total_words': 18146},
 'char_metrics': {'substitutions': 7496,
  'insertions': 8308,
  'deletions': 32524,
  'total_chars': 89719}}

A disclaimer for the Wrist Angel Dataset is that the utterance-based segments were constructed from taking the Ground-Truth transcripts and cutting the audio at the timestamps that were given from these transcripts.
However, one drawback to this approach, and which might lead to an artifically high error rate on any model tested on the Wrist Angel audio segments, are that the ground-truth transcripts only contained start times for each utterance.
This means that a choice was made to segment the utterances from the start time of the current utterance to the start time of the next utterance.
Upon further investigation into the quality of the transcripts, it was found that certain boundaries did not match exactly, giving in incorrect matching between the transcript and the clipped segment.

Therefore, to truly assess the models abilities to accurately transcribe the audio from the Wrist Angel dataset, it will also evaluate their accuracy-based performance on processing the full audio-clips, from which the segments originated.

## Computing Sentence Semantic Distance
As part of the transcript accuracy evaluation, I will compute a more context-based accuracy evaluation in order to provide a more nuanced perspective on the correctness of the generated transcripts.
It will be using a standard multilingual sentence transformer for all SemDist evaluations to ensure consistency in the results.

In [4]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util

def compute_sentence_semantic_distance(
    references: list[str], 
    hypotheses: list[str], 
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
) -> list[float]:
    """
    Calculates the Sentence Semantic Distance (1 - Cosine Similarity) 
    between reference and ASR hypothesis transcripts.
    """
    # Load Sentence Transformer model
    model = SentenceTransformer(model_name)
    
    # 1. Compute dense embeddings for references and hypotheses
    ref_embeddings = model.encode(references, convert_to_tensor=True, show_progress_bar=False)
    hyp_embeddings = model.encode(hypotheses, convert_to_tensor=True, show_progress_bar=False)
    
    # 2. Calculate pairwise cosine similarity along the diagonal
    # util.cos_sim returns a similarity matrix of shape [N, N]
    cosine_sims = util.cos_sim(ref_embeddings, hyp_embeddings).diagonal()
    
    # 3. Convert similarity to distance: Distance = 1 - Similarity
    semantic_distances = 1.0 - cosine_sims.cpu().numpy()
    
    # Clip small negative floating-point errors around zero
    semantic_distances = np.clip(semantic_distances, 0.0, 2.0)
    
    return semantic_distances.tolist()


### Using Sentence-based Semantic Distance for evaluating the Roest-v3-Wav2Vec2-315m Models Performance on CoRal

In [ ]:
wav2vec2_hyp = []
wav2vec2_ref = []
with open("../results/wav2vec2_baseline.jsonl", "r", encoding="utf-8") as file:
    iter = 1
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            # Look for epoch metadata (adjust key to match your logger)
            if iter >= 1689:
                wav2vec2_hyp.append(sample['hyp'])
                wav2vec2_ref.append(sample['ref'])
        iter += 1

distances = compute_sentence_semantic_distance(references=wav2vec2_ref, hypotheses=wav2vec2_hyp)
for r, h, d in zip(wav2vec2_ref, wav2vec2_hyp, distances):
    print(f"REF: {r}")
    print(f"HYP: {h}")
    print(f"Semantic Distance: {d:.4f}\n")

### Using Sentence-based Semantic Distance for evaluating the Roest-v3-Wav2Vec2-315m Model on the Wrist Angel Dataset

In [5]:
wav2vec2_hyp = []
wav2vec2_ref = []
with open("../results/wa_wav2vec2_baseline.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            wav2vec2_hyp.append(sample['hyp'])
            wav2vec2_ref.append(sample['ref'])

distances = compute_sentence_semantic_distance(references=wav2vec2_ref, hypotheses=wav2vec2_hyp)
for r, h, d in zip(wav2vec2_ref, wav2vec2_hyp, distances):
    print(f"REF: {r}")
    print(f"HYP: {h}")
    print(f"Semantic Distance: {d:.4f}\n")

c:\Users\ATHO0358\Documents\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ATHO0358\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not in

REF: Og når man prøver at arbejde mod OCDen så i virkeligheden, så skal man prøve at vælge et eller andet, man kan prøve at gøre og mærke noget ubehag. Det kan godt være lidt svært at sådan... Men når man får de her tanker, som kan være ubehagelige eller som man ikke rigtig kan slippe af med, kan du så mærke ubehaget i kroppen? Der er nogle, der kan mærke det ved, at deres puls stiger eller de får svært ved at koncentrere sig eller de får det bare rigtig ubehageligt. Hvis du skulle sige nu bare af at snakke om hvor ubehageligt du havde det ved at vi sidder og snakker om det her, hvor ville det så være henne?
HYP: arbejde mod usibien så e i virkeligheden så skal man prøve at vælge et eller andet man kan prøve at gøre og mærke noget ubehag det kan jo godt være lidt svært og sådan men når man får de her tanker soman kan være ubehagelige eller som man ikke rigtig kan slippe af noget kan du så mærke uehagen i kroppen der er nogen der kan mærke det ved at deres pulstier eller de forsvært ved

In [7]:
print(len(distances))
print(np.average(distances))

3138
0.3794127168018701


### Using Sentence-based Semantic Distance for evaluating the 8-bit Quantized Roest-v3-Whisper-1.5b Model on the Wrist Angel Dataset

In [8]:
ct2_hyp = []
ct2_ref = []
with open("../results/wa_ct2_int8.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            ct2_hyp.append(sample['hyp'])
            ct2_ref.append(sample['ref'])

distances_ct2 = compute_sentence_semantic_distance(references=ct2_ref, hypotheses=ct2_hyp)
for r, h, d in zip(ct2_ref, ct2_hyp, distances_ct2):
    print(f"REF: {r}")
    print(f"HYP: {h}")
    print(f"Semantic Distance: {d:.4f}\n")

REF: Og når man prøver at arbejde mod OCDen så i virkeligheden, så skal man prøve at vælge et eller andet, man kan prøve at gøre og mærke noget ubehag. Det kan godt være lidt svært at sådan... Men når man får de her tanker, som kan være ubehagelige eller som man ikke rigtig kan slippe af med, kan du så mærke ubehaget i kroppen? Der er nogle, der kan mærke det ved, at deres puls stiger eller de får svært ved at koncentrere sig eller de får det bare rigtig ubehageligt. Hvis du skulle sige nu bare af at snakke om hvor ubehageligt du havde det ved at vi sidder og snakker om det her, hvor ville det så være henne?
HYP: . . Ja.
Semantic Distance: 0.8702

REF: 10.
HYP: 10
Semantic Distance: 0.0294

REF: Okay. Så du kan mærke i kroppen, at det er rigtig ubehageligt?
HYP: Okay. Så du kan mærke i kroppen det er rigtig ubehageligt?
Semantic Distance: 0.0034

REF: Ja.
HYP: Og når man har det så ubehageligt
Semantic Distance: 0.5457

REF: Og når man har det så ubehageligt, så plejer vi faktisk bare 

In [9]:
print(np.average(distances_ct2))

0.3346491260743026


These results of evaluating the two models on the "wild" audio from the Wrist Angel dataset, show that both models might struggle with the natural language of a free-flowing conversation between multiple individuals. Furthermore, the small difference in performance between the models, i.e. 0.33 error rate for the 8-bit quantized whisper model and the 0.38 error for the ctc-based wav2vec2 model, lead me to suspect that attaching a lightweight language model to the ctc-based model could correct some of the misspellings observed in the resulting transcripts, which may have improved the overall error rate of the models performance.

# For evaluating speaker-based accuracy

In [ ]:
from collections import defaultdict
import jiwer

def evaluate_speaker_level_asr(dataset: list[dict]) -> dict[str, dict]:
    """
    Computes corpus-level (micro-averaged) WER, CER, and detailed error 
    breakdowns for each individual speaker in the dataset.
    
    `dataset` should be a list of dicts formatted as:
      [{"speaker_id": "spk_01", "reference": "...", "hypothesis": "..."}, ...]
    """
    # 1. Group transcripts by speaker_id
    speaker_data = defaultdict(lambda: {"refs": [], "hyps": []})
    for entry in dataset:
        spk = entry["speaker_id"]
        speaker_data[spk]["refs"].append(entry["reference"])
        speaker_data[spk]["hyps"].append(entry["hypothesis"])

    speaker_results = {}

    # 2. Compute micro-averaged metrics for each speaker
    for spk_id, data in speaker_data.items():
        refs = data["refs"]
        hyps = data["hyps"]

        # Word-level (WER)
        w_out = jiwer.process_words(refs, hyps)
        w_sub, w_ins, w_del = w_out.substitutions, w_out.insertions, w_out.deletions
        w_total = w_out.hits + w_sub + w_del

        # Character-level (CER)
        c_out = jiwer.process_characters(refs, hyps)
        c_sub, c_ins, c_del = c_out.substitutions, c_out.insertions, c_out.deletions
        c_total = c_out.hits + c_sub + c_del

        speaker_results[spk_id] = {
            "num_utterances": len(refs),
            "wer": w_out.wer,
            "cer": c_out.cer,
            "word_breakdown": {
                "total_words": w_total,
                "substitutions": w_sub,
                "insertions": w_ins,
                "deletions": w_del,
            },
            "char_breakdown": {
                "total_chars": c_total,
                "substitutions": c_sub,
                "insertions": c_ins,
                "deletions": c_del,
            }
        }

    return speaker_results


def print_speaker_summary(speaker_results: dict[str, dict]):
    """Utility to print a clean summary table of speaker results."""
    print(f"{'Speaker ID':<15} | {'Utterances':<10} | {'WER':<8} | {'CER':<8} | {'Word Errors (S/I/D)':<20}")
    print("-" * 72)
    
    for spk_id, res in speaker_results.items():
        w_b = res["word_breakdown"]
        errors_str = f"{w_b['substitutions']}/{w_b['insertions']}/{w_b['deletions']}"
        print(f"{spk_id:<15} | {res['num_utterances']:<10} | {res['wer']:>6.2%} | {res['cer']:>6.2%} | {errors_str:<20}")




In [19]:
from collections import defaultdict
import jiwer

def evaluate_speaker_level_asr(dataset: list[dict]) -> dict[str, dict]:

    speaker_data = defaultdict(lambda: {"refs": [], "hyps": []})
    for entry in dataset:
        spk = entry["speaker_id"]
        speaker_data[spk]["refs"].append(entry["ref"])
        speaker_data[spk]["hyps"].append(entry["hyp"])

    speaker_results = {}

    for spk_id, data in speaker_data.items():
        refs = data["refs"]
        hyps = data["hyps"]

        # WER
        w_out = jiwer.process_words(refs, hyps)
        w_sub, w_ins, w_del = w_out.substitutions, w_out.insertions, w_out.deletions
        w_total = w_out.hits + w_sub + w_del

        # CER
        c_out = jiwer.process_characters(refs, hyps)
        c_sub, c_ins, c_del = c_out.substitutions, c_out.insertions, c_out.deletions
        c_total = c_out.hits + c_sub + c_del

        speaker_results[spk_id] = {
            "num_utterances": len(refs),
            "wer": w_out.wer,
            "cer": c_out.cer,
            "word_breakdown": {
                "total_words": w_total,
                "substitutions": w_sub,
                "insertions": w_ins,
                "deletions": w_del,
            },
            "char_breakdown": {
                "total_chars": c_total,
                "substitutions": c_sub,
                "insertions": c_ins,
                "deletions": c_del,
            }
        }

    return speaker_results


def print_speaker_summary(speaker_results: dict[str, dict]):
    print(f"{'Speaker ID':<15} | {'Utterances':<10} | {'WER':<8} | {'CER':<8} | {'Word Errors (S/I/D)':<20}")
    print("-" * 72)
    
    for spk_id, res in speaker_results.items():
        w_b = res["word_breakdown"]
        errors_str = f"{w_b['substitutions']}/{w_b['insertions']}/{w_b['deletions']}"
        print(f"{spk_id:<15} | {res['num_utterances']:<10} | {res['wer']:>6.2%} | {res['cer']:>6.2%} | {errors_str:<20}")




### Evaluating Speaker-based Performance on CoRal Dataset

In [26]:
from datasets import Dataset

ds = load_dataset("CoRal-project/coral-v3", "conversation", split='test')
df = Dataset.to_pandas(ds)
id_to_speaker = df.set_index('id_conversation')['id_speaker'].to_dict()
all_samples = []
with open("../results/wav2vec2_baseline.jsonl") as file:
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            audio_id = sample.get('audio_id') 
            # Map speaker_id using dictionary lookup (defaults to None if not found)
            sample['id_speaker'] = id_to_speaker.get(audio_id, None)
            
            all_samples.append(sample)


In [ ]:
speaker_dict = evaluate_speaker_level_asr(all_samples)
print_speaker_summary(speaker_dict)

Speaker ID      | Utterances | WER      | CER      | Word Errors (S/I/D) 
------------------------------------------------------------------------
spe_fa639f5932359117682753884585d883 | 390        | 24.94% | 12.24% | 1088/246/502        
spe_5e319f90767d47e11731d95e314e4670 | 732        | 44.33% | 20.09% | 1786/292/276        
spe_df3293886215084f5fd6a447bb379b11 | 402        | 31.51% | 13.51% | 768/152/152         
spe_4aa23a60464a18e3597cdeb3606ac572 | 1188       | 34.32% | 16.60% | 2648/458/710        
spe_03e8b9d0ee8d3192e113ff62c61e4916 | 812        | 36.24% | 16.82% | 2578/350/756        
spe_deedf738efa054ae460989be3033a3cf | 238        | 23.42% | 10.10% | 410/102/102         
spe_741ba3dd1acd26458718a591a980d743 | 638        | 29.14% | 12.99% | 1802/268/420        
spe_2937b289da4c0a7b9877c56ecead4794 | 566        | 29.38% | 13.27% | 1604/266/342        
spe_fbf3381f525dbe5ddf1a2a1d36e9c4b9 | 756        | 42.45% | 20.10% | 2048/304/388        
spe_6e67cbe51a49d9e4abbd7699a4a89d

In [ ]:
ds = load_dataset("CoRal-project/coral-v3", "conversation", split='test')
df = Dataset.to_pandas(ds)
id_to_speaker = df.set_index('id_conversation')['id_speaker'].to_dict()
all_samples = []
with open("../results/wav2vec2_baseline.jsonl") as file:
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            audio_id = sample.get('audio_id') 
            speaker_id = id_to_speaker.get(audio_id, None)
            speaker_err = speaker_dict.get(speaker_id)
            sample['wer'] = speaker_err['wer']
            sample['cer'] = speaker_err['cer']
            all_samples.append(sample)

### Evaluating Speaker-based Performance of the Roest-v3-Wav2Vec2-315m Model on Wrist Angel Dataset

In [21]:
import json

df = pd.read_csv('..\\data\\audio\\segments\\metadata_new.csv')
id_to_speaker_id = df.set_index('segment_id')['speaker'].to_dict()

samples = []
with open("../results/wa_wav2vec2_baseline.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            audio_id = sample.get("segment_id")
            sample["speaker_id"] = id_to_speaker_id.get(audio_id, None)
            samples.append(sample)

speaker_results = evaluate_speaker_level_asr(samples)
print_speaker_summary(speaker_results=speaker_results)

Speaker ID      | Utterances | WER      | CER      | Word Errors (S/I/D) 
------------------------------------------------------------------------
I               | 1458       | 66.22% | 38.24% | 11589/2673/4407     
B               | 1260       | 84.59% | 55.68% | 7605/2829/3156      
V               | 405        | 76.22% | 48.23% | 2775/531/1530       


### Evaluating Speaker-based Performance of the CT2 8-Bit Quantized Roest-v3-Whisper-1.5b Model on Wrist Angel Dataset

In [22]:
import json

df = pd.read_csv('..\\data\\audio\\segments\\metadata_new.csv')
id_to_speaker_id = df.set_index('segment_id')['speaker'].to_dict()

samples = []
with open("../results/wa_ct2_int8.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            audio_id = sample.get("segment_id")
            sample["speaker_id"] = id_to_speaker_id.get(audio_id, None)
            samples.append(sample)

speaker_results = evaluate_speaker_level_asr(samples)
print_speaker_summary(speaker_results=speaker_results)

Speaker ID      | Utterances | WER      | CER      | Word Errors (S/I/D) 
------------------------------------------------------------------------
I               | 1560       | 58.95% | 38.56% | 8829/3426/4704      
B               | 1482       | 79.58% | 54.80% | 6861/4065/2388      
V               | 471        | 75.84% | 50.11% | 2676/1161/1182      


From the results above, it can be seen that the speaker type which both models appear to struggle the least with recognizing is Speaker I. This was expected given that the interviewer (Speaker I) is usually speaking in a slower pace and with clearer pronounciation in addition to also experiencing fewer interruptions during speech segments.
One noticable thing is that for the type of Word Errors, the substitution errors was the predominant struggle for the ctc-based wav2vec2 model, with a noticable higher word substitution error rate for the interviewer (Speaker I) than for the remaining speaker types. However, for the quantized whisper model, it showed that while the substitution errors were still the dominant one of the word errors, the insertion-related word errors resulted in a vast increase. While the insertion word errors only increased 28% for Speaker I when testing with the Whisper model, the insertion errors for Speaker B (adolescent speakers) saw an increase of 43.7% (nearly doubled).

## Calculating and evaluating the computational resource related metrics
For this section I will run through computing the different metrics, including; CPU Time, Wall Time, RAM usage, Real-Time-Factor (RTF), Latency and Throughput.

The RAM usage will more so be reported as a descriptive statistic than an actual evaluation. Furthermore, the CPU efficiency will also be estimated in order to assess whether there was/is room for potential improvements if the core utilization was not maximized.

In [11]:
def get_walltime(filename: str):
    epoch_pattern = r'Epoch: (\d+)'
    walltime_pattern = r'Walltime:\s*([+-]?(?:[0-9]*\.)?[0-9]+)'
    cputime_pattern = r'CPU time:\s*([+-]?(?:[0-9]*\.)?[0-9]+)'

    walltime_res = []
    cputime_res = []
    with open(filename, 'r') as file:
        for line in file:
            m_epoch = re.search(epoch_pattern, line)
            m_walltime = re.search(walltime_pattern, line)
            m_cputime = re.search(cputime_pattern, line)

            if not m_epoch or not m_walltime or not m_cputime:
                continue


            walltime = float(m_walltime.group(1))
            cpu_time = float(m_cputime.group(1))
            walltime_res.append(walltime)
            cputime_res.append(cpu_time)

    return walltime_res, cputime_res


In [12]:
def get_durations(filename: str):
    durations = []
    with open(filename, 'r') as file:
        for line in file:
            batch = json.loads(line)
            batch_duration = 0
            for sample in batch:
                sample_duration = sample['seg_end']
                batch_duration += sample_duration
            durations.append(batch_duration)

    return durations

In [14]:
import pandas as pd

def compute_performance_metrics(
    df: pd.DataFrame, 
    num_cpu_threads: int = 6
) -> pd.DataFrame:
    """
    Computes Wall Time, CPU Time, RTF, CPU Parallelization Efficiency, 
    and RAM Savings relative to a specified baseline model
    """
    # Group metrics by model (averaging across logged runs/batches)
    summary = df.groupby('model_name').agg({
        'wall_time_sec': 'sum',
        'cpu_time_sec': 'sum',
        'total_audio_duration_sec': 'sum'
    }).reset_index()

    # 1. Compute Real-Time Factor (RTF)
    summary['RTF'] = summary['wall_time_sec'] / summary['total_audio_duration_sec']

    # 2. Compute CPU Parallelization Efficiency (%)
    # Ratio of CPU time spent relative to theoretical maximum across allocated threads
    summary['CPU_Efficiency_pct'] = (
        summary['cpu_time_sec'] / (summary['wall_time_sec'] * num_cpu_threads)
    ) * 100


    return summary

### Overall Compute Performance Metrics
#### Roest-v3-wav2vec2-315m - CoRal

In [ ]:
walltime_arr, cputime_arr = get_walltime('../results/wav2vec2_baseline.log')
durations = get_durations('../results/wav2vec2_baseline.jsonl')
model_name = ['roest-v3-wav2vec2-315m'] * len(walltime_arr)
df_compute = pd.DataFrame({'model_name': model_name,'wall_time_sec': walltime_arr, 'cpu_time_sec': cputime_arr, 'total_audio_duration_sec': durations})

metrics_summary = compute_performance_metrics(
    df_compute, 
    num_cpu_threads=6
)

# Display formatted table
print(metrics_summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

#### Roest-v3-wav2vec2-315m - Wrist Angel

In [17]:
walltime_arr, cputime_arr = get_walltime('../results/wav2vec2_baseline_logs.log')
durations = get_durations('../results/wa_wav2vec2_baseline.jsonl')
model_name = ['roest-v3-wav2vec2-315m'] * len(walltime_arr)

print(len(walltime_arr))
print(len(cputime_arr))
print(len(durations))
df_compute = pd.DataFrame({'model_name': model_name,'wall_time_sec': walltime_arr, 'cpu_time_sec': cputime_arr, 'total_audio_duration_sec': durations})

metrics_summary = compute_performance_metrics(
    df_compute, 
    num_cpu_threads=6
)

# Display formatted table
print(metrics_summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

627
627
627
            model_name  wall_time_sec  cpu_time_sec  total_audio_duration_sec  RTF  CPU_Efficiency_pct
roest-v3-wav2vec2-315m        8111.39      48029.94                  27678.00 0.29               98.69


#### 8-bit Quantized CT2 version of Roest-v3-Whisper-1.5b - Wrist Angel

In [18]:
walltime_arr, cputime_arr = get_walltime('../results/ct2_baseline_logs.log')
durations = get_durations('../results/wa_ct2_int8.jsonl')
model_name = ['roest-v3-whisper-1.5b-ct2-int8'] * len(walltime_arr)

print(len(walltime_arr))
print(len(cputime_arr))
print(len(durations))
df_compute = pd.DataFrame({'model_name': model_name,'wall_time_sec': walltime_arr, 'cpu_time_sec': cputime_arr, 'total_audio_duration_sec': durations})

metrics_summary = compute_performance_metrics(
    df_compute, 
    num_cpu_threads=6
)

# Display formatted table
print(metrics_summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

705
705
705
                    model_name  wall_time_sec  cpu_time_sec  total_audio_duration_sec  RTF  CPU_Efficiency_pct
roest-v3-whisper-1.5b-ct2-int8       50653.13     300339.12                  28068.00 1.80               98.82


From the computational resource metrics shown above, it is clear that there is a vast difference in how fast the models process audio files. Although they both utilized the CPU cores efficiently, the ctc-based model was more than x6 times faster than the seq2seq-based model in terms of their real-time-factor (i.e. the time it takes to process a single audio file relative to the duration of the single audio file, so RTF > 1.0 means that to process an audio file takes longer than the duration of the file itself)

# Carbon Footprint Results:

## Testing CT2 8-Bit Quantized Roest-v3-Whisper-1.5b Model
CarbonTracker:

Actual consumption for 3 epoch(s):

        Time:   14:07:47

        Energy: 0.795035695076 kWh

        CO2eq:  113.928949019396 g

        This is equivalent to:
        
        1.066750458983 km travelled by car

CarbonTracker: Finished monitoring.